In [ ]:
import os
import json
from PIL import Image
from pathlib import Path

In [ ]:
import os
import cv2
import itertools
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence

# --- 1. 기본 설정 ---
BASE_DATA_PATH = './245.개인 특정을 위한 자필과 모사 필기체 데이터/01-1.정식개방데이터'
IMG_SIZE = (96, 96)
SEQ_LENGTH = 150 # 시계열 이미지 최대 길이
BATCH_SIZE = 4
NUM_LABEL_FEATURES = 2 

# --- 2. 데이터 경로 파싱 및 페어 생성 함수 (수정됨) ---
def create_data_pairs(data_type='Training'):
    """
    (최종) '자필' 내 다른 자세=Positive, '자필'vs'모사' 동일 자세=Negative로 페어를 생성합니다.
    (수정: 지정된 자세/번호만 필터링하여 로드)
    """
    source_path = os.path.join(BASE_DATA_PATH, data_type, '01.원천데이터')
    if data_type == 'Validation':
        source_path = os.path.join(source_path, 'VS')

    pairs = []
    labels = []
    
    sentence_folders = [f'문장{i:02d}' for i in range(1, 6)]

    for sent_folder in sentence_folders:
        sent_path = os.path.join(source_path, sent_folder)
        if not os.path.exists(sent_path):
            continue

        user_signatures = {}
        for signature_folder in os.listdir(sent_path):
            parts = signature_folder.split('_')
            if len(parts) < 3: continue
            
            sig_type, user_id = parts[0], parts[-1]
            if user_id not in user_signatures:
                user_signatures[user_id] = {}
            
            user_signatures[user_id][sig_type] = os.path.join(sent_path, signature_folder)

        # --- 새로운 페어링 로직 ---
        for user_id, data in user_signatures.items():
            if '자필' in data and '모사' in data:
                genuine_folder = data['자필']
                forged_folder = data['모사']

                # ⭐️⭐️⭐️ [수정된 부분 시작] ⭐️⭐️⭐️
                # 1. 사용자가 지정한 자세와 번호 목록을 정의합니다.
                actions_to_use = ['앉아서', '일어서서']  # '벽에기대서'는 제외
                indices_to_use = range(1, 4)           # 1, 2, 3번만 사용

                # 허용된 폴더 이름 집합 생성 (예: {'앉아서_01', '일어서서_03', ...})
                allowed_actions = {f'{action}_{i:02d}' for action in actions_to_use for i in indices_to_use}

                # 2. 디스크에 실제 존재하는 자세 폴더 중, 위에서 허용한 폴더만 필터링합니다.
                all_disk_actions = [d for d in os.listdir(genuine_folder) if os.path.isdir(os.path.join(genuine_folder, d))]
                available_actions = [action for action in all_disk_actions if action in allowed_actions]
                # ⭐️⭐️⭐️ [수정된 부분 끝] ⭐️⭐️⭐️
                
                # 이제 available_actions 에는 '앉아서_01~03', '일어서서_01~03' 중 실제 존재하는 폴더만 들어있습니다.
                
                # 1. Positive Pairs 생성 (자필 폴더 내 필터링된 자세끼리 비교)
                if len(available_actions) >= 2:
                    for action1, action2 in itertools.combinations(available_actions, 2):
                        path1 = os.path.join(genuine_folder, action1)
                        path2 = os.path.join(genuine_folder, action2)
                        pairs.append((path1, path2))
                        labels.append(1.0) # 동일인이므로 1.0

                # 2. Negative Pairs 생성 (자필 vs 모사 동일 자세 비교)
                for action in available_actions:
                    anchor_path = os.path.join(genuine_folder, action)
                    negative_path = os.path.join(forged_folder, action)
                    
                    if os.path.exists(negative_path):
                        pairs.append((anchor_path, negative_path))
                        labels.append(0.0) # 타인이므로 0.0
    
    return pairs, labels

# --- 3. Keras 데이터 제너레이터 (변경 없음) ---
# (이 부분은 이전과 동일하므로, 설명을 위해 생략합니다. 실제 코드에는 포함되어야 합니다.)
class SignatureDataGenerator(Sequence):
    def __init__(self, pairs, labels, batch_size):
        self.pairs = pairs
        self.labels = labels
        self.batch_size = batch_size
        self.indices = np.arange(len(self.pairs))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.pairs) / self.batch_size))

    def __getitem__(self, index):
        start_index = index * self.batch_size
        end_index = (index + 1) * self.batch_size
        batch_indices = self.indices[start_index:end_index]
        
        current_batch_size = len(batch_indices)
        X1 = np.zeros((current_batch_size, SEQ_LENGTH, *IMG_SIZE, 1))
        X2 = np.zeros((current_batch_size, SEQ_LENGTH, *IMG_SIZE, 1))
        
        batch_pairs = [self.pairs[i] for i in batch_indices]
        y = np.array([self.labels[i] for i in batch_indices])

        for i, (path1, path2) in enumerate(batch_pairs):
            X1[i,] = self._load_sequence(path1)
            X2[i,] = self._load_sequence(path2)
            
        return [X1, X2], y

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

    def _load_sequence(self, folder_path):
        if not os.path.exists(folder_path):
            return np.zeros((SEQ_LENGTH, *IMG_SIZE, 1))
            
        image_files = sorted([os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.png')])
        sequence = np.zeros((SEQ_LENGTH, *IMG_SIZE, 1))

        for i, img_path in enumerate(image_files):
            if i >= SEQ_LENGTH:
                break
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, IMG_SIZE)
                img = img / 255.0  # Normalize
                sequence[i, :, :, 0] = img
        return sequence

# --- 4. 데이터 로드 실행 ---
print("학습 데이터 쌍 생성 중...")
train_pairs, train_labels = create_data_pairs('Training')
print("검증 데이터 쌍 생성 중...")
val_pairs, val_labels = create_data_pairs('Validation')

print(f"총 학습용 페어: {len(train_pairs)}")
print(f"총 검증용 페어: {len(val_pairs)}")

if len(train_pairs) > 0:
    train_generator = SignatureDataGenerator(train_pairs, train_labels, BATCH_SIZE)
    val_generator = SignatureDataGenerator(val_pairs, val_labels, BATCH_SIZE)
    print("✅ 데이터 제너레이터 생성 완료!")
else:
    print("❌ 생성된 데이터 페어가 없습니다. BASE_DATA_PATH와 하위 폴더 구조를 다시 확인해주세요.")

In [ ]:
import os
import cv2
import json
import itertools
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence

# --- 1. 기본 설정 ---
BASE_DATA_PATH = './245.개인 특정을 위한 자필과 모사 필기체 데이터/01-1.정식개방데이터'
IMG_SIZE = (128, 128)
SEQ_LENGTH = 150 # 시계열 이미지 최대 길이
BATCH_SIZE = 16
# ⭐️ JSON에서 추출할 특징(feature)의 개수. 예: 평균 속도, 평균 필압 등 2개
NUM_LABEL_FEATURES = 2 

# --- 2. 데이터 경로 파싱 및 페어 생성 함수 ---
def create_data_pairs(data_type='Training'):
    """
    (최종) '자필' 내 다른 자세=Positive, '자필'vs'모사' 동일 자세=Negative로 페어를 생성합니다.
    """
    source_path = os.path.join(BASE_DATA_PATH, data_type, '01.원천데이터')
    if data_type == 'Validation':
        source_path = os.path.join(source_path, 'VS')

    pairs = []
    labels = []
    
    sentence_folders = [f'문장{i:02d}' for i in range(1, 6)]

    for sent_folder in sentence_folders:
        sent_path = os.path.join(source_path, sent_folder)
        if not os.path.exists(sent_path):
            continue

        user_signatures = {}
        for signature_folder in os.listdir(sent_path):
            parts = signature_folder.split('_')
            if len(parts) < 3: continue
            
            sig_type, user_id = parts[0], parts[-1]
            if user_id not in user_signatures:
                user_signatures[user_id] = {}
            
            user_signatures[user_id][sig_type] = os.path.join(sent_path, signature_folder)

        for user_id, data in user_signatures.items():
            if '자필' in data and '모사' in data:
                genuine_folder = data['자필']
                forged_folder = data['모사']
                available_actions = [d for d in os.listdir(genuine_folder) if os.path.isdir(os.path.join(genuine_folder, d))]
                
                if len(available_actions) >= 2:
                    for action1, action2 in itertools.combinations(available_actions, 2):
                        pairs.append((os.path.join(genuine_folder, action1), os.path.join(genuine_folder, action2)))
                        labels.append(1.0)

                for action in available_actions:
                    anchor_path = os.path.join(genuine_folder, action)
                    negative_path = os.path.join(forged_folder, action)
                    if os.path.exists(negative_path):
                        pairs.append((anchor_path, negative_path))
                        labels.append(0.0)
    
    return pairs, labels

# --- 3. Keras 데이터 제너레이터 (JSON 로딩 기능 추가) ---
class SignatureDataGenerator(Sequence):
    def __init__(self, pairs, labels, batch_size):
        self.pairs = pairs
        self.labels = labels
        self.batch_size = batch_size
        self.indices = np.arange(len(self.pairs))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.pairs) / self.batch_size))

    def __getitem__(self, index):
        start_index = index * self.batch_size
        end_index = (index + 1) * self.batch_size
        batch_indices = self.indices[start_index:end_index]
        
        current_batch_size = len(batch_indices)
        # ⭐️ 4개의 입력을 준비: 이미지(X1, X2), 라벨(L1, L2)
        X1_img = np.zeros((current_batch_size, SEQ_LENGTH, *IMG_SIZE, 1))
        X2_img = np.zeros((current_batch_size, SEQ_LENGTH, *IMG_SIZE, 1))
        L1_label = np.zeros((current_batch_size, NUM_LABEL_FEATURES))
        L2_label = np.zeros((current_batch_size, NUM_LABEL_FEATURES))
        
        y = np.array([self.labels[i] for i in batch_indices])

        for i, pair_idx in enumerate(batch_indices):
            path1, path2 = self.pairs[pair_idx]
            X1_img[i,] = self._load_sequence(path1)
            X2_img[i,] = self._load_sequence(path2)
            # ⭐️ 라벨링 데이터 로딩 추가
            L1_label[i,] = self._load_label_data(path1)
            L2_label[i,] = self._load_label_data(path2)
            
        # ⭐️ 모델에 전달할 입력이 4개가 되므로 리스트 형태로 반환
        return (X1_img, L1_label, X2_img, L2_label), y

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

    def _load_sequence(self, folder_path):
        if not os.path.exists(folder_path): return np.zeros((SEQ_LENGTH, *IMG_SIZE, 1))
        image_files = sorted([os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.png')])
        sequence = np.zeros((SEQ_LENGTH, *IMG_SIZE, 1))
        for i, img_path in enumerate(image_files):
            if i >= SEQ_LENGTH: break
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, IMG_SIZE)
                sequence[i, :, :, 0] = img / 255.0
        return sequence

    # ⭐️ JSON 데이터를 로드하는 새로운 함수
    def _load_label_data(self, image_folder_path):
        try:
            # 이미지 경로를 라벨링 데이터 경로로 변환
            label_path = image_folder_path.replace('01.원천데이터', '02.라벨링데이터')
            if 'VS' in label_path:
                label_path = label_path.replace('VS', 'VL')
            
            # 자세 폴더명(예: 앉아서_01)에 .json 확장자를 붙여 파일 경로 생성
            json_filename = os.path.basename(label_path) + '.json'
            # 최종 JSON 경로는 자세 폴더의 부모 디렉토리 + 파일명
            json_path = os.path.join(os.path.dirname(label_path), json_filename)

            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # ❗️❗️ JSON 파일의 실제 키(key)에 맞게 이 부분을 수정해야 합니다. ❗️❗️
            # 예시: 'properties' 안에 'average_speed' 와 'average_pressure' 가 있다고 가정
            feature1 = data['properties']['average_speed']
            feature2 = data['properties']['average_pressure']
            
            return np.array([feature1, feature2])
        except Exception:
            # JSON 파일이 없거나 키가 다른 경우 0으로 채움
            return np.zeros(NUM_LABEL_FEATURES)

# --- 4. 데이터 로드 실행 ---
print("학습 데이터 쌍 생성 중...")
train_pairs, train_labels = create_data_pairs('Training')
print("검증 데이터 쌍 생성 중...")
val_pairs, val_labels = create_data_pairs('Validation')

print(f"총 학습용 페어: {len(train_pairs)}")
print(f"총 검증용 페어: {len(val_pairs)}")

if len(train_pairs) > 0:
    train_generator = SignatureDataGenerator(train_pairs, train_labels, BATCH_SIZE)
    val_generator = SignatureDataGenerator(val_pairs, val_labels, BATCH_SIZE)
    print("✅ 데이터 제너레이터 생성 완료!")

    # --- 5. 최종 데이터 로딩 확인 ---
    print("\n--- ✅ 데이터 로딩 샘플 확인 ---")
    if len(train_generator) > 0:
        # 제너레이터에서 첫 번째 배치를 가져옴
        [batch_X1_img, batch_L1_label, batch_X2_img, batch_L2_label], batch_y = train_generator[0]
        
        print(f"배치 크기: {len(batch_y)}")
        print(f"이미지 배치 1 shape: {batch_X1_img.shape}")
        print(f"라벨 배치 1 shape: {batch_L1_label.shape}")
        print(f"이미지 배치 2 shape: {batch_X2_img.shape}")
        print(f"라벨 배치 2 shape: {batch_L2_label.shape}")
        print("-" * 30)
        print(f"첫 번째 페어의 이미지1 데이터 일부 (첫 프레임의 좌상단 5x5):")
        print(batch_X1_img[0, 0, :5, :5, 0])
        print(f"첫 번째 페어의 라벨1 데이터 샘플: {batch_L1_label[0]}")
        print(f"첫 번째 페어의 정답 (1.0=동일인, 0.0=타인): {batch_y[0]}")
    else:
        print("❌ 확인할 학습 데이터 배치가 없습니다.")
else:
    print("❌ 생성된 데이터 페어가 없습니다.")

In [ ]:
######### 1단계: 샴 네트워크 모델 구축
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, TimeDistributed, Conv2D, MaxPooling2D, Flatten, Bidirectional, LSTM, Dense, Lambda, Concatenate
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# --- 모델 구축 ---

def build_base_network(image_shape, label_shape):
    """
    이미지 시퀀스와 라벨링 데이터를 입력받아 하나의 특징 벡터(embedding)를 출력하는 기본 네트워크
    """
    # 2개의 입력을 받는 구조
    image_input = Input(shape=image_shape, name="image_input")
    label_input = Input(shape=label_shape, name="label_input")

    # A. 이미지 처리 파이프라인 (CNN -> BiLSTM)
    x = TimeDistributed(Conv2D(32, (3, 3), activation='relu'), name='conv1')(image_input)
    x = TimeDistributed(MaxPooling2D((2, 2)))(x)
    x = TimeDistributed(Conv2D(64, (3, 3), activation='relu'), name='conv2')(x)
    x = TimeDistributed(MaxPooling2D((2, 2)))(x)
    x = TimeDistributed(Flatten())(x)
    image_features = Bidirectional(LSTM(64))(x)

    # B. 라벨링 데이터 처리 파이프라인 (Dense Layers)
    y = Dense(16, activation='relu')(label_input)
    label_features = Dense(32, activation='relu')(y)

    # C. 이미지 특징과 라벨링 특징을 결합
    combined_features = Concatenate()([image_features, label_features])
    
    # 최종 임베딩 벡터 생성
    embedding = Dense(128, activation='relu')(combined_features)
    
    return Model(inputs=[image_input, label_input], outputs=embedding, name="BaseNetwork")

def contrastive_loss(y_true, y_pred):
    """
    샴 네트워크 학습을 위한 대조 손실 함수
    - 동일한 쌍(y_true=1)의 거리는 가깝게 만들고, 다른 쌍(y_true=0)의 거리는 멀게 만듦
    """
    margin = 1.0
    square_pred = K.square(y_pred)
    margin_square = K.square(K.maximum(margin - y_pred, 0))
    return K.mean(y_true * square_pred + (1 - y_true) * margin_square)

def euclidean_distance(vectors):
    """두 벡터 간의 유클리드 거리를 계산하는 함수"""
    x, y = vectors
    sum_square = K.sum(K.square(x - y), axis=1, keepdims=True)
    return K.sqrt(K.maximum(sum_square, K.epsilon()))


# --- 전체 샴 네트워크 모델 조립 ---
image_input_shape = (SEQ_LENGTH, *IMG_SIZE, 1)
label_input_shape = (NUM_LABEL_FEATURES,)

# 1. Base Network 생성
base_network = build_base_network(image_input_shape, label_input_shape)

# 2. 샴 네트워크의 두 '팔'에 해당하는 4개의 입력 레이어 정의
input_a_img = Input(shape=image_input_shape, name="anchor_image")
input_a_label = Input(shape=label_input_shape, name="anchor_label")
input_b_img = Input(shape=image_input_shape, name="comparison_image")
input_b_label = Input(shape=label_input_shape, name="comparison_label")

# 3. 각 입력 쌍을 동일한 Base Network에 통과시켜 임베딩 벡터 추출
embedding_a = base_network([input_a_img, input_a_label])
embedding_b = base_network([input_b_img, input_b_label])

# 4. 두 임베딩 벡터 간의 거리 계산
distance = Lambda(euclidean_distance)([embedding_a, embedding_b])

# 5. 최종 모델 정의
siamese_model = Model(
    inputs=[input_a_img, input_a_label, input_b_img, input_b_label], 
    outputs=distance
)

# 6. 모델 컴파일
siamese_model.compile(loss=contrastive_loss, optimizer='adam')

print("✅ 모델 구축 완료!")
siamese_model.summary()

In [ ]:
# ######## 2단계: 모델 학습

# --- 모델 학습 시작 ---

# 학습 중 가장 성능이 좋은 모델만 저장
checkpoint = ModelCheckpoint(
    "best_siamese_model.keras",
    save_best_only=True, 
    monitor="val_loss", 
    mode="min",
    verbose=1
)

# 5 에포크 동안 성능 개선이 없으면 학습 조기 종료
early_stopping = EarlyStopping(
    monitor="val_loss", 
    patience=5, 
    mode="min",
    verbose=1
)

# 데이터가 있을 경우에만 학습 진행
if len(train_generator) > 0:
    print("\n--- 훈련을 시작합니다 ---")
    history = siamese_model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=30,  # 최대 에포크 수
        callbacks=[checkpoint, early_stopping],
        verbose=1  # ⭐️ 이 옵션을 추가하면 진행률과 결과가 출력됩니다.
    )
    print("\n✅ 모델 학습 완료!")

    # 학습 과정 시각화 (선택 사항)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

else:
    print("❌ 학습할 데이터가 없습니다.")

In [ ]:
######## 3단계: 결과 예측 및 열지도(Heatmap). 시각화
# --- 학습된 모델 로드 및 예측/시각화 ---

# 가장 성능이 좋았던 모델 가중치 불러오기
print("최적의 모델 'best_siamese_model.h5'를 로드합니다.")
siamese_model.load_weights("best_siamese_model.h5")

def get_gradcam_heatmap(model, image_seq, last_conv_layer_name):
    """
    시계열 이미지에 대한 Grad-CAM 열지도를 생성하는 함수
    """
    grad_model = Model(
        model.inputs, 
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image_seq)
        # 임베딩 벡터의 L2 Norm을 손실로 사용하여 그래디언트 계산
        loss = tf.norm(predictions)

    grads = tape.gradient(loss, conv_outputs)
    
    # 각 프레임별로 Heatmap 생성
    all_heatmaps = []
    for i in range(SEQ_LENGTH):
        frame_conv_output = conv_outputs[0, i]
        frame_grads = grads[0, i]
        
        pooled_grads = tf.reduce_mean(frame_grads, axis=(0, 1))
        heatmap = tf.einsum('ij,j->ij', frame_conv_output, pooled_grads)
        heatmap = tf.reduce_sum(heatmap, axis=-1)
        heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + K.epsilon())
        all_heatmaps.append(heatmap.numpy())
        
    return np.array(all_heatmaps)

def generate_heatmap_explanation(heatmap_sequence):
    """
    열지도를 분석하여 일반 사용자를 위한 간단한 설명을 생성하는 함수
    """
    # 가장 활성화가 높았던 프레임과 위치 찾기
    avg_activation_per_frame = np.mean(heatmap_sequence, axis=(1, 2))
    peak_frame_idx = np.argmax(avg_activation_per_frame)
    
    peak_heatmap = heatmap_sequence[peak_frame_idx]
    y, x = np.unravel_index(np.argmax(peak_heatmap), peak_heatmap.shape)
    
    # 시간적 위치 분석
    if peak_frame_idx < SEQ_LENGTH / 3:
        timing = "초반"
    elif peak_frame_idx > SEQ_LENGTH * 2 / 3:
        timing = "후반"
    else:
        timing = "중반"
        
    # 공간적 위치 분석
    location = ""
    if y < IMG_SIZE[0] / 3: location += "상단 "
    elif y > IMG_SIZE[0] * 2 / 3: location += "하단 "
    else: location += "중앙 "
    if x < IMG_SIZE[1] / 3: location += "좌측"
    elif x > IMG_SIZE[1] * 2 / 3: location += "우측"
    
    explanation = (
        f"주석: 모델은 필기 과정의 '{timing}' 단계에 가장 주목했습니다. "
        f"특히 서명의 '{location}' 영역에서 나타나는 고유한 필기 습관(예: 획의 방향, 필압)을 "
        "본인 여부를 판단하는 핵심 근거로 사용한 것으로 보입니다."
    )
    return explanation

# --- 시각화 실행 ---
# 검증 데이터에서 샘플 하나를 가져와 테스트
[test_a_img, test_a_label, test_b_img, test_b_label], test_y = val_generator[0]

sample_idx = 0  # 첫 번째 샘플을 사용
anchor_img = test_a_img[sample_idx:sample_idx+1]
anchor_label = test_a_label[sample_idx:sample_idx+1]
compare_img = test_b_img[sample_idx:sample_idx+1]
compare_label = test_b_label[sample_idx:sample_idx+1]
ground_truth = "동일인" if test_y[sample_idx] == 1.0 else "다른 사람(모사)"

# 1. 유사도 예측
distance = siamese_model.predict([anchor_img, anchor_label, compare_img, compare_label])[0][0]
threshold = 0.5  # 임계값 (실제로는 검증 데이터로 최적값을 찾아야 함)
prediction = "동일인" if distance < threshold else "다른 사람(모사)"

print(f"\n--- 예측 결과 ---")
print(f"계산된 거리: {distance:.4f} (낮을수록 유사)")
print(f"모델 예측: {prediction} (실제: {ground_truth})")

# 2. 열지도(Heatmap) 생성
base_model_for_gradcam = siamese_model.get_layer('BaseNetwork')
# 마지막 Conv2D 레이어의 이름 (model.summary()에서 확인)
last_conv_layer_name = 'conv2' 

heatmap_seq = get_gradcam_heatmap(base_model_for_gradcam, anchor_img, last_conv_layer_name)
explanation = generate_heatmap_explanation(heatmap_seq)

# 3. 결과 시각화
num_frames_to_show = 5
indices = np.linspace(0, SEQ_LENGTH - 1, num_frames_to_show, dtype=int)

fig, axes = plt.subplots(2, num_frames_to_show, figsize=(18, 8))
fig.suptitle(f"Prediction: {prediction} (Actual: {ground_truth}) | Distance: {distance:.4f}", fontsize=16)

for i, frame_idx in enumerate(indices):
    # 원본 이미지
    axes[0, i].imshow(anchor_img[0, frame_idx, ..., 0], cmap='gray')
    axes[0, i].set_title(f"Original Frame {frame_idx}")
    axes[0, i].axis('off')

    # 열지도가 덧씌워진 이미지
    heatmap = cv2.resize(heatmap_seq[frame_idx], IMG_SIZE)
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    original_rgb = cv2.cvtColor(np.uint8(anchor_img[0, frame_idx, ..., 0] * 255), cv2.COLOR_GRAY2RGB)
    superimposed_img = heatmap * 0.4 + original_rgb * 0.6
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)

    axes[1, i].imshow(superimposed_img)
    axes[1, i].set_title(f"Heatmap Frame {frame_idx}")
    axes[1, i].axis('off')

plt.figtext(0.5, 0.05, explanation, ha="center", fontsize=12, wrap=True, bbox={"facecolor":"white", "alpha":0.7, "pad":5})
plt.tight_layout(rect=[0, 0.1, 1, 0.95])
plt.show()

In [ ]:
import os
import cv2
import itertools
import numpy as np
import gc # 가비지 컬렉션을 위한 라이브러리

# --- 1. 기본 설정 (이전과 동일) ---
BASE_DATA_PATH = './245.개인 특정을 위한 자필과 모사 필기체 데이터/01-1.정식개방데이터'
IMG_SIZE = (96, 96)
SEQ_LENGTH = 150
BATCH_SIZE = 4

# --- 2. 데이터 경로 파싱 및 페어 생성 함수 (이전과 동일) ---
def create_data_pairs(data_type='Training'):
    source_path = os.path.join(BASE_DATA_PATH, data_type, '01.원천데이터')
    if data_type == 'Validation':
        source_path = os.path.join(source_path, 'VS')
    pairs, labels = [], []
    sentence_folders = [f'문장{i:02d}' for i in range(1, 6)]
    for sent_folder in sentence_folders:
        sent_path = os.path.join(source_path, sent_folder)
        if not os.path.exists(sent_path): continue
        user_signatures = {}
        for signature_folder in os.listdir(sent_path):
            parts = signature_folder.split('_')
            if len(parts) < 3: continue
            sig_type, user_id = parts[0], parts[-1]
            if user_id not in user_signatures: user_signatures[user_id] = {}
            user_signatures[user_id][sig_type] = os.path.join(sent_path, signature_folder)
        for user_id, data in user_signatures.items():
            if '자필' in data and '모사' in data:
                genuine_folder, forged_folder = data['자필'], data['모사']
                actions_to_use = ['앉아서', '일어서서']
                indices_to_use = range(1, 4)
                allowed_actions = {f'{action}_{i:02d}' for action in actions_to_use for i in indices_to_use}
                all_disk_actions = [d for d in os.listdir(genuine_folder) if os.path.isdir(os.path.join(genuine_folder, d))]
                available_actions = [action for action in all_disk_actions if action in allowed_actions]
                if len(available_actions) >= 2:
                    for action1, action2 in itertools.combinations(available_actions, 2):
                        pairs.append((os.path.join(genuine_folder, action1), os.path.join(genuine_folder, action2)))
                        labels.append(1.0)
                for action in available_actions:
                    anchor_path = os.path.join(genuine_folder, action)
                    negative_path = os.path.join(forged_folder, action)
                    if os.path.exists(negative_path):
                        pairs.append((anchor_path, negative_path))
                        labels.append(0.0)
    return pairs, labels

# --- 3. Keras 데이터 제너레이터 (Sequence 임포트 제외, 여기선 직접 사용) ---
class TestSignatureDataGenerator: # tf.keras.utils.Sequence 상속 제외
    def __init__(self, pairs, labels, batch_size):
        self.pairs, self.labels, self.batch_size = pairs, labels, batch_size
        self.indices = np.arange(len(self.pairs))

    def __len__(self):
        return int(np.ceil(len(self.pairs) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        X1 = np.zeros((len(batch_indices), SEQ_LENGTH, *IMG_SIZE, 1))
        X2 = np.zeros((len(batch_indices), SEQ_LENGTH, *IMG_SIZE, 1))
        for i, pair_idx in enumerate(batch_indices):
            path1, path2 = self.pairs[pair_idx]
            X1[i,] = self._load_sequence(path1)
            X2[i,] = self._load_sequence(path2)
        # 실제 데이터만 반환, y값은 테스트에 불필요
        return X1, X2
        
    def _load_sequence(self, folder_path):
        if not os.path.exists(folder_path): return np.zeros((SEQ_LENGTH, *IMG_SIZE, 1))
        image_files = sorted([os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.png')])
        sequence = np.zeros((SEQ_LENGTH, *IMG_SIZE, 1))
        for i, img_path in enumerate(image_files):
            if i >= SEQ_LENGTH: break
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, IMG_SIZE)
                sequence[i, :, :, 0] = img / 255.0
        return sequence

# --- 4. 데이터 로드 및 제너레이터 생성 ---
print("학습 데이터 쌍 생성 중...")
train_pairs, train_labels = create_data_pairs('Training')
print(f"총 학습용 페어: {len(train_pairs)}")

if len(train_pairs) > 0:
    test_generator = TestSignatureDataGenerator(train_pairs, train_labels, BATCH_SIZE)
    print("✅ 테스트 제너레이터 생성 완료!")

    # --- 5. 제너레이터 순회 테스트 ---
    print("\n--- 순수 데이터 로딩 테스트 시작 ---")
    try:
        # 처음 5개의 배치만 로드해보고 문제가 없는지 확인
        for i in range(len(test_generator)):
            if i >= 5: # 너무 오래 걸리지 않게 5 배치만 테스트
                print("\n✅ 처음 5개 배치 로딩 성공!")
                break
            
            print(f"배치 {i + 1}/{len(test_generator)} 로딩 중...")
            batch_X1, batch_X2 = test_generator[i]
            print(f" -> 배치 {i + 1} 로딩 완료 (X1 shape: {batch_X1.shape}, X2 shape: {batch_X2.shape})")
            
            # 메모리 누수를 방지하기 위해 가비지 컬렉션 강제 호출
            gc.collect()

        print("\n--- ✅ 순수 데이터 로딩 테스트 성공 ---")

    except Exception as e:
        print(f"\n--- ❌ 데이터 로딩 중 오류 발생 ---")
        print(f"오류: {e}")

else:
    print("❌ 생성된 데이터 페어가 없습니다.")